------------------------------
#### Embedding as a text feature encoder for ML algorithms
------------------------------

In [4]:
import pandas as pd
import numpy as np
from ast import literal_eval

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, confusion_matrix, classification_report

In [5]:
datafile_path = r"D:\Makesh\Working\AI\RPS\Day07\Dataset\FoodReview\amazon_food_reviews_with_embeddings_2k.csv"

In [6]:
df = pd.read_csv(datafile_path)

In [7]:
df.shape

(2000, 9)

In [8]:
df.sample(3)

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,n_tokens,ada_embedding
1431,420982,B008RMU23S,A1JW7RH2UKUXW6,5,best ever,The type of food stuff that calls to you in th...,Title: best ever; Content: The type of food st...,37,"[-0.012798125855624676, -0.011409515514969826,..."
383,450464,B00876U540,A3PJURDRBMF2QM,5,Good for the price!,"My family like this apple jacks cereal, and it...",Title: Good for the price!; Content: My family...,40,"[-0.006442038342356682, -0.02990194968879223, ..."
1494,551923,B0062P9XPU,A33KQALCZGXG8C,5,Delicious!,I am not a huge beer lover. I do enjoy an occ...,Title: Delicious!; Content: I am not a huge be...,97,"[0.008289854042232037, 0.010335138067603111, -..."


**about literal_eval**

In [9]:
# Example 1: Safely evaluating a numeric literal
number_str = "42"
number = literal_eval(number_str)
print(number)  # Output: 42

# Example 2: Safely evaluating a list literal
list_str = "[1, 2, 3, 4]"
my_list = literal_eval(list_str)
print(my_list)  # Output: [1, 2, 3, 4]

# Example 3: Safely evaluating a dictionary literal
dict_str = "{'key': 'value', 'number': 123}"
my_dict = literal_eval(dict_str)
print(my_dict)  # Output: {'key': 'value', 'number': 123}

42
[1, 2, 3, 4]
{'key': 'value', 'number': 123}


In [10]:
df.dtypes

Unnamed: 0        int64
ProductId        object
UserId           object
Score             int64
Summary          object
Text             object
combined         object
n_tokens          int64
ada_embedding    object
dtype: object

In [11]:
df["embedding"] = df.ada_embedding.apply(literal_eval).apply(np.array)

In [12]:
df.dtypes

Unnamed: 0        int64
ProductId        object
UserId           object
Score             int64
Summary          object
Text             object
combined         object
n_tokens          int64
ada_embedding    object
embedding        object
dtype: object

In [13]:
df.sample(2)

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,n_tokens,ada_embedding,embedding
26,459089,B002OFU7W0,A11WY5413ZACZR,1,"Disgustingly salty, in a dog treat shape",I wanted to get some jerky and since I like bu...,"Title: Disgustingly salty, in a dog treat shap...",90,"[0.015635592862963676, 0.009291375987231731, -...","[0.015635592862963676, 0.009291375987231731, -..."
669,61534,B005FT1JTW,AD30TCZOL5F1N,5,"Love it, or not so much","As my title says, you either will love it or n...","Title: Love it, or not so much; Content: As my...",60,"[0.009893771260976791, -0.00360675947740674, -...","[0.009893771260976791, -0.00360675947740674, -..."


In [14]:
list(df.embedding.values)[:3]

[array([ 0.02544181, -0.02061664, -0.03099807, ..., -0.00384917,
         0.00072606, -0.01538206], shape=(1536,)),
 array([ 0.01369932,  0.02614656, -0.03358544, ..., -0.00362185,
         0.02311208, -0.0047653 ], shape=(1536,)),
 array([ 0.03073794, -0.00048475, -0.00901532, ...,  0.02461325,
        -0.00506217, -0.00576694], shape=(1536,))]

In [15]:
list(df.embedding.values)[0].shape

(1536,)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(list(df.embedding.values), 
                                                        df.Score, 
                                                        test_size   = 0.2, 
                                                        random_state= 42)

**Logistic regression**

In [17]:
logr = LogisticRegression()
logr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [18]:
preds = logr.predict(X_test)

In [19]:
accuracy_score(y_test, preds), confusion_matrix(y_test, preds)

(0.805,
 array([[ 29,   0,   0,   1,   3],
        [  6,   3,   3,   1,   1],
        [  2,   1,  14,   5,  11],
        [  0,   0,   1,  11,  38],
        [  0,   0,   0,   5, 265]]))

In [20]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           1       0.78      0.88      0.83        33
           2       0.75      0.21      0.33        14
           3       0.78      0.42      0.55        33
           4       0.48      0.22      0.30        50
           5       0.83      0.98      0.90       270

    accuracy                           0.81       400
   macro avg       0.72      0.54      0.58       400
weighted avg       0.78      0.81      0.77       400



In [21]:
# mse = mean_squared_error(y_test, preds)
# mae = mean_absolute_error(y_test, preds)

# print(f"ada-002 embedding performance on 2k Amazon reviews: mse={mse:.2f}, mae={mae:.2f}")

We can see that the embeddings are able to predict the scores with an average error of 0.53 per score prediction. This is roughly equivalent to predicting half of reviews perfectly, and half off by one star.